# 🛡️ Day-4 · Notebook 04: Grounded Generation & Hallucination Guardrails
**KIT Campus RAG Agent · Part 4: Context Injection, Citations & Negative Constraints**

In this notebook, we build:
1. **The Grounded Context Prompt**: Injecting retrieved chunks with explicit citation tags (`[Doc 1]`, `[Doc 2]`).
2. **Mandatory Citations**: Forcing the model to attribute every fact to a source document.
3. **Defensive Hallucination Guardrails**: Instructing the model to state *"Insufficient context"* when asked questions not covered by the official handbook (e.g., *"What is the college fee for B.Tech CSE?"*).
4. **Gemini 3.7 / 3.6 Flash Integration**: Calling modern Gemini models with offline grounded fallback.



In [ ]:
from typing import List, Dict

def build_grounded_prompt(query: str, retrieved_docs: List[Dict[str, str]]) -> str:
    context_blocks = []
    for i, doc in enumerate(retrieved_docs, start=1):
        context_blocks.append(f"""[Doc {i}] (Source: {doc['source']}, Section: {doc['section']})
{doc['content']}""")
    
    context_str = "\n\n".join(context_blocks)
    
    prompt = f"""You are the official KIT College of Engineering AI Academic Advisor.
Answer the student's question strictly using ONLY the provided official reference documents below.

CRITICAL RULES:
1. Every factual assertion must be tagged with its source bracket reference, e.g. [Doc 1].
2. If the answer cannot be determined strictly from the provided documents, explicitly state:
   "I do not have sufficient information in the official documents to answer this question."
3. Do NOT invent policies, fees, dates, or contact details.

OFFICIAL REFERENCE CONTEXT:
{context_str}

STUDENT QUESTION:
{query}

OFFICIAL GROUNDED ANSWER:"""
    return prompt

sample_docs = [
    {
        "source": "kit_college_student_handbook.md",
        "section": "1.2 Medical Condonation",
        "content": "Students with attendance between 65.0% and 74.9% may apply for medical condonation by submitting a medical certificate within 3 days and paying ₹500 fee per subject."
    },
    {
        "source": "kit_placement_policy_guide.pdf",
        "section": "Placement Eligibility",
        "content": "Minimum 7.00 CGPA with no standing arrears is required for campus placement registration."
    }
]

prompt = build_grounded_prompt("Can I apply for condonation if my attendance is 68%?", sample_docs)
print(prompt)



In [ ]:
# Simulating defensive guardrails on out-of-context questions
import re

def verify_citations(answer: str, max_docs: int) -> bool:
    """Checks if the response contains valid bracket citations like [Doc 1]."""
    matches = re.findall(r"\[Doc\s+(\d+)\]", answer)
    if not matches:
        return False
    doc_ids = [int(m) for m in matches]
    return all(1 <= d <= max_docs for d in doc_ids)

valid_answer = "Yes, you can apply for medical condonation since your attendance of 68% is between 65.0% and 74.9% [Doc 1]. You must submit a doctor certificate within 3 days and pay ₹500 fee per subject [Doc 1]."
invalid_answer = "Yes, you can apply for condonation by paying ₹500 to the office."

print(f"Valid Answer Verified:   {verify_citations(valid_answer, max_docs=2)}")
print(f"Invalid Answer Verified: {verify_citations(invalid_answer, max_docs=2)}")

